<a href="https://colab.research.google.com/github/Kevin-AngelQD/padp/blob/main/Sesion10_271758.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Kevin Angel Quiñonez Dominguez

**Matrícula:** 271758

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [1]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [2]:
# Tu código aquí

#df_marketing.columns
import re

df_marketing_renombrado = df_marketing.copy() #Copia de los datos originales
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower() #minusculas
df_marketing_renombrado.rename(columns={'kidhome': 'kid_home', 'mntwines': 'mnt_wines','mntfruits': 'mnt_fruits'}, inplace=True) #Se renombraron  unos para mostrar un ejemplo pero se realiza automatizado en lambda

df_marketing_renombrado = df_marketing_renombrado.rename(
    columns=lambda x: re.sub(
        r'(prods|products)$', r'_\1',
        re.sub(r'^mnt([a-z])', r'mnt_\1', x.lower())
    )
)

df_marketing_renombrado.columns




Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teenhome', 'dt_customer', 'recency', 'mnt_wines', 'mnt_fruits',
       'mnt_meat_products', 'mnt_fish_products', 'mnt_sweet_products',
       'mnt_gold_prods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [3]:
df_netflix['date_added'].isnull().sum()

np.int64(10)

In [4]:
# Tu código aquí
df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')
df_netflix.dtypes
df_netflix['date_added'].isnull().sum()


np.int64(10)

In [5]:
#Parece no tener diferencias de antes o despues respecto a nulos.
df_netflix['date_added']

,date_added
0,2020-08-14
1,2016-12-23
2,2018-12-20
3,2017-11-16
4,2020-01-01
...,...
7782,2020-10-19
7783,2019-03-02
7784,2020-09-25
7785,2020-10-31


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [6]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [7]:
# Tu código aquí

print("Duplicados determinados con sum: ",df_marketing_dup.duplicated().sum()) #duplicados con sum
print("Duplicados determinados con ID: ",df_marketing_dup.duplicated(subset='ID').sum()) #duplicados con ID
df_marketing_dup.drop_duplicates(inplace=True)
df_marketing_dup.shape #Sin duplicados


Duplicados determinados con sum:  2
Duplicados determinados con ID:  2


(2240, 29)

---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [8]:
df_netflix.isnull().sum()#Valores nulos totales de cada columna


,0
show_id,0
type,0
title,0
director,2389
cast,718
country,507
date_added,10
release_year,0
rating,7
duration,0


In [9]:
df_netflix[df_netflix.isnull().any(axis=1)] #Filas con algun nulo

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
11,s12,TV Show,1983,NaN,"Robert Więckiewicz, Maciej Musiał, Michalina O...","Poland, United States",2018-11-30,2018,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Dramas","In this dark alt-history thriller, a naïve law..."
12,s13,TV Show,1994,Diego Enrique Osorno,NaN,Mexico,2019-05-17,2019,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...",Archival video and new interviews examine Mexi...
16,s17,TV Show,09-Feb,NaN,"Shahd El Yaseen, Shaila Sabt, Hala, Hanadi Al-...",NaN,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas","As a psychology professor faces Alzheimer's, h..."
19,s20,Movie,'89,NaN,"Lee Dixon, Ian Wright, Paul Merson",United Kingdom,2018-05-16,2017,TV-PG,87 min,Sports Movies,"Mixing old footage with interviews, this is th..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7777,s7778,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01,2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
7779,s7780,TV Show,Zona Rosa,NaN,"Manu NNa, Ana Julia Yeyé, Ray Contreras, Pablo...",Mexico,2019-11-26,2019,TV-MA,1 Season,"International TV Shows, Spanish-Language TV Sh...",An assortment of talent takes the stage for a ...
7784,s7785,Movie,Zulu Man in Japan,NaN,Nasty C,NaN,2020-09-25,2019,TV-MA,44 min,"Documentaries, International Movies, Music & M...","In this documentary, South African rapper Nast..."
7785,s7786,TV Show,Zumbo's Just Desserts,NaN,"Adriano Zumbo, Rachel Khoo",Australia,2020-10-31,2019,TV-PG,1 Season,"International TV Shows, Reality TV",Dessert wizard Adriano Zumbo looks for the nex...


In [10]:
# Tu código aquí
df_netflix.isnull().any(axis=1).sum() #Filas con nulos

np.int64(2979)

---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [11]:
# Tu código aquí

(1 - df_netflix.isnull().sum()) / len(df_netflix) * 100 #Completitud en porcentaje

#director seria la mas baja, incluso esto se podia predecir de cierta forma ya que era la columna con mas valores nulos en Actividad 4



,0
show_id,0.012842
type,0.012842
title,0.012842
director,-30.666495
cast,-9.207654
country,-6.498010
date_added,-0.115577
release_year,0.012842
rating,-0.077051
duration,0.012842


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [12]:
# Tu código aquí

df_marketing['Marital_Status'].value_counts() #Ver categorias

#Solo reclasificaria alone en single, absurd y yolo pueden significar diferentes cosas a las cuales no tenemos contexto
#(broma entre pareja o trolling al sistema en el cuestionario) por lo que considero viable descartar esos datos

,count
Marital_Status,
Married,864
Together,580
Single,480
Divorced,232
Widow,77
Alone,3
Absurd,2
YOLO,2


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [13]:
# Tu código aquí
df_netflix['show_id'].str.match(r'^s\d+$').mean() * 100 #100% de cumplimiento

np.float64(100.0)

---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [14]:
# Tu código aquí
df_marketing['Year_Birth'].describe()


,Year_Birth
count,2240.000000
mean,1968.805804
std,11.984069
min,1893.000000
25%,1959.000000
50%,1970.000000
75%,1977.000000
max,1996.000000


In [15]:
df_marketing.nsmallest(5, 'Year_Birth')
#Tomando en cuenta su fecha en dt_customer tomaria en cuenta que es un error de dedo.

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
239,11004,1893,2n Cycle,Single,60182.0,0,1,2014-05-17,23,8,...,4,0,0,0,0,0,0,3,11,0
339,1150,1899,PhD,Together,83532.0,0,0,2013-09-26,36,755,...,1,0,0,1,0,0,0,3,11,0
192,7829,1900,2n Cycle,Divorced,36640.0,1,0,2013-09-26,99,15,...,5,0,0,0,0,0,1,3,11,0
1950,6663,1940,PhD,Single,51141.0,0,0,2013-07-08,96,144,...,5,0,0,0,0,0,0,3,11,0
424,6932,1941,PhD,Married,93027.0,0,0,2013-04-13,77,1285,...,2,0,0,1,0,0,0,3,11,0


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [16]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [17]:
# Paso 1 — ajuste de tipos


df_practica.dtypes
#Income se volvio objeto por el sesenta mil en texto agregado, citando el refran mexicano "Todos coludos o todos rabones" en terminos de python si hay un coludo, todos los rabones se les tratara como coludo

,0
ID,int64
Year_Birth,int64
Education,object
Marital_Status,object
Income,object
Kidhome,int64
Teenhome,int64
Dt_Customer,object
Recency,int64
MntWines,int64


In [18]:
df_practica['Income'] = pd.to_numeric(df_marketing['Income'], errors='coerce') #Ajuste de tipos

df_practica.dtypes

,0
ID,int64
Year_Birth,int64
Education,object
Marital_Status,object
Income,float64
Kidhome,int64
Teenhome,int64
Dt_Customer,object
Recency,int64
MntWines,int64


In [19]:
# Paso 2 — duplicados

print(df_practica.duplicated().sum()) #Duplicados dentro


df_practica.drop_duplicates(inplace=True) #Duplicados fuera
df_practica.shape

0


(16, 29)

In [20]:
# Paso 3 — valores faltantes
df_practica.isnull().sum() #Valores nulos totales

,0
ID,0
Year_Birth,0
Education,0
Marital_Status,0
Income,1
Kidhome,0
Teenhome,0
Dt_Customer,0
Recency,0
MntWines,0


In [21]:
# Paso 4 — exploración categórica
df_practica['Marital_Status'].value_counts() #Valores unicos
#Tomando en cuenta el contexto de si Together es que no estan casados pero estan juntos talvez utilizar un termino similar para union libre (cohabitation por ejemplo),
#Ya que no sabemos de esos 7 quienes de verdad esten casados legalmente

,count
Marital_Status,
Together,7
Married,5
Single,3
Divorced,1


**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*

Determino que los datos en general pueden mostrar informacion importante en ciertos aspectos, estan considerablemente limpios pero algunos tienen informacion faltante  como lo es la seccion de directores en el dataset de netflix u erronea como en marital status en marketing con ciertos valores que no cumplen las caracteristicas. La mayoria puede tener un uso si se descartan o corrigen los datos que muestren problemas dandoles el formato adecuado para su uso.  